# 03 — Approach A: Improved zero-shot LLM

Port of the Go classifier with three key changes:

1. **4-category LLM schema** (`junk | service_feedback | config_feedback | app_specific`). `others` is not an LLM output; it is the rule-based fallback/source bucket for rows the rules could not cover and that the LLM must reclassify.
2. **XML-structured prompt** (V4 in `shared/prompts.py`) — clearer signal than the V3 labelled-line format for small models.
3. **Slim agent context** — uses cached `agentSummary`, `services` as `name:endpoint` flat strings, OASF top-K shallow paths only.

Benchmarks:
- `rule_based`: held-out records whose labels came from rule-based categories.
- `hand_labelled`: manually labelled records from `scripts/labelled/others_gold_v1.csv`.

Same model panel as the Go bench:
`qwen2.5:3b`, `qwen2.5:7b-instruct`, `qwen3:8b`, `gemma3:12b`, `qwen2.5:14b-instruct`, `phi4`

Output: `data/results/approach_a_{bench}_{model}.parquet` per model and benchmark + a combined `approach_a_all.parquet`.

In [1]:
import sys, json
from pathlib import Path
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from tqdm.auto import tqdm
from shared.mongo_client import fetch_agents_by_keys
from shared.ollama_client import OllamaClient
from shared.prompts import system_prompt_v4
from shared.context_builder import build_user_message
from shared.types import AgentMeta, FeedbackRecord
from shared.metrics import per_class_report, confusion_df, summarize_run

SPLITS = AI_ROOT / 'data' / 'splits'
BENCHES = {
    'rule_based': pd.read_parquet(SPLITS / 'rule_based' / 'test.parquet'),
    'hand_labelled': pd.read_parquet(SPLITS / 'hand_labelled' / 'test.parquet'),
}
for bench, df in BENCHES.items():
    print(f'{bench:13s}', len(df), dict(df['label'].value_counts()))

rule_based    1023 {'config_feedback': np.int64(300), 'app_specific': np.int64(300), 'service_feedback': np.int64(300), 'junk': np.int64(123)}
hand_labelled 197 {'service_feedback': np.int64(138), 'app_specific': np.int64(36), 'config_feedback': np.int64(22), 'junk': np.int64(1)}


In [2]:
# Bulk-load agent metadata (one Mongo round-trip) for both benchmark sets.
all_tests = pd.concat(BENCHES.values(), ignore_index=True)
keys = list(set((int(r.chain_id), str(r.agent_id)) for r in all_tests.itertuples()))
agent_docs = fetch_agents_by_keys(keys)
print('agents loaded:', len(agent_docs))

def parse_services(raw_services: str) -> list[dict]:
    if not isinstance(raw_services, str) or not raw_services.strip():
        return []
    try:
        data = json.loads(raw_services)
    except json.JSONDecodeError:
        return []
    return data if isinstance(data, list) else []

def parse_tags(raw_tags: str) -> list[str]:
    if not isinstance(raw_tags, str) or not raw_tags.strip():
        return []
    return [x.strip() for x in raw_tags.replace('|', ',').split(',') if x.strip()]

def agent_meta_for(row) -> AgentMeta:
    doc = agent_docs.get(f'{row.chain_id}:{row.agent_id}', {})
    csv_description = getattr(row, 'agent_description', '') or ''
    csv_services = getattr(row, 'agent_services', '') or ''
    csv_tags = getattr(row, 'agent_tags', '') or ''
    description = doc.get('description', '') or csv_description
    return AgentMeta(
        chain_id=int(row.chain_id), agent_id=str(row.agent_id),
        name=doc.get('name','') or f'agent:{row.agent_id}',
        description=description,
        summary=doc.get('agentSummary','') or csv_description or description,
        services=doc.get('services', []) or parse_services(csv_services),
        oasf_domains=doc.get('oasfDomains', []) or [],
        oasf_skills=doc.get('oasfSkills', []) or [],
        tags=doc.get('tags', []) or parse_tags(csv_tags),
    )

def feedback_record_for(row) -> FeedbackRecord:
    fp = row.feedback_parsed
    if isinstance(fp, str) and fp.strip().startswith('{'):
        try: fp = json.loads(fp)
        except json.JSONDecodeError: fp = None
    else:
        fp = None
    return FeedbackRecord(
        id=row.id, agent_id=str(row.agent_id), chain_id=int(row.chain_id),
        tag1=row.tag1 or '', tag2=row.tag2 or '',
        endpoint=row.endpoint or '', value=str(row.value),
        value_decimals=int(row.value_decimals), value_scale=row.value_scale or '',
        feedback_parsed=fp,
        rule_category=getattr(row, 'rule_category', row.label),
        is_self_feedback=bool(row.is_self_feedback),
    )

agents loaded: 844


In [3]:
# Pre-build prompts once (model-agnostic) — saves repeating the same string concat per model pass.
prompts_by_bench: dict[str, list[tuple[str, str, str]]] = {}
for bench, df in BENCHES.items():
    prompts = []
    for r in df.itertuples():
        fb = feedback_record_for(r)
        a = agent_meta_for(r)
        prompts.append((r.id, r.label, build_user_message(a, fb)))
    prompts_by_bench[bench] = prompts
    print(f'{bench:13s} prompts:', len(prompts))

print('\n--- sample ---')
print(prompts_by_bench['rule_based'][0][2])

rule_based    prompts: 1023
hand_labelled prompts: 197

--- sample ---
<agent>
  <name>Chronos Vault</name>
  <summary>DeFi: Chronos Vault preserves the exact state of February 26, 2026 in digital archives for historical preservation.</summary>
  <services>web:https://celonova.xyz/agent/4959 | api:https://api.celonova.xyz/api/agents/4959 | OASF:https://github.com/agntcy/oasf/ | mcp:https://api.celonova.xyz/mcp | a2a:https://api.celonova.xyz/.well-known/agent-card.json</services>
  <domains>finance/global_economics, geopolitics/international_relations</domains>
  <skills>analytical_skills/geopolitical_analysis, analytical_skills/world_events, information_skills/news_synthesis</skills>
</agent>
<feedback>
  <tag1>trust</tag1>
  <tag2>oracle-screening</tag2>
  <endpoint>Oracle screening: risk=low, flags=[UNVERIFIED]</endpoint>
  <scale>pct100</scale>
  <value>85</value>
</feedback>


In [4]:
# Model panel. Edit MODELS to subset while iterating. Smaller models include
# few-shot examples in the system prompt; ≥12B run without (cheaper context).
MODELS = [
    ('qwen2.5:3b',           True),
    ('qwen2.5:7b-instruct',  True),
    ('qwen3:8b',             True),
    ('gemma3:12b',           False),
    ('qwen2.5:14b-instruct', False),
    ('phi4',                 False),
]

RESULTS_DIR = AI_ROOT / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
all_runs = []
for bench, prompts in prompts_by_bench.items():
    for model_name, few_shot in MODELS:
        print(f'\n=== {bench} / {model_name} (few_shot={few_shot}) ===')
        slug = model_name.replace(':','_').replace('/','_')
        out_path = RESULTS_DIR / f'approach_a_{bench}_{slug}.parquet'
        if out_path.exists():
            print('  → cached, skipping. Delete', out_path.name, 'to re-run.')
            all_runs.append(pd.read_parquet(out_path))
            continue

        rows = []
        sys_prompt = system_prompt_v4(include_few_shot=few_shot)
        with OllamaClient(model=model_name, system_prompt=sys_prompt) as cli:
            if not cli.health():
                print('  ⚠ Ollama unreachable; skipping')
                continue
            for fid, label, user_msg in tqdm(prompts, desc=bench):
                res = cli.classify(user_msg)
                rows.append({
                    'id': fid, 'bench': bench, 'model': model_name, 'gold': label,
                    'pred': res.category, 'confidence': res.confidence,
                    'reason': res.reason, 'latency_ms': res.latency_ms,
                    'source': res.source,
                })
        run_df = pd.DataFrame(rows)
        run_df.to_parquet(out_path, index=False)
        all_runs.append(run_df)
        print('  wrote', out_path, len(run_df), 'rows')


=== rule_based / qwen2.5:3b (few_shot=True) ===


rule_based:   0%|          | 0/1023 [00:00<?, ?it/s]

  wrote /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/results/approach_a_rule_based_qwen2.5_3b.parquet 1023 rows

=== rule_based / qwen2.5:7b-instruct (few_shot=True) ===


rule_based:   0%|          | 0/1023 [00:00<?, ?it/s]

  wrote /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/results/approach_a_rule_based_qwen2.5_7b-instruct.parquet 1023 rows

=== rule_based / qwen3:8b (few_shot=True) ===


...

  wrote /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/results/approach_a_hand_labelled_gemma3_12b.parquet 197 rows

=== hand_labelled / qwen2.5:14b-instruct (few_shot=False) ===


hand_labelled:   0%|          | 0/197 [00:00<?, ?it/s]

  wrote /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/results/approach_a_hand_labelled_qwen2.5_14b-instruct.parquet 197 rows

=== hand_labelled / phi4 (few_shot=False) ===


hand_labelled:   0%|          | 0/197 [00:00<?, ?it/s]

  wrote /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/results/approach_a_hand_labelled_phi4.parquet 197 rows


In [6]:
# Per-model summary, sorted by benchmark then macro-F1 desc.
#
# STANDALONE — loads from data/results/ so you can re-evaluate metrics
# without re-running the LLM cells (no Mongo, no Ollama needed).
#
# Macro-F1 is computed over the 4 real categories only:
# junk / service_feedback / config_feedback / app_specific.

import sys
from pathlib import Path
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from shared.metrics import summarize_run

RESULTS_DIR = AI_ROOT / 'data' / 'results'
per_model_files = sorted(
    p
    for bench in ('rule_based', 'hand_labelled')
    for p in RESULTS_DIR.glob(f'approach_a_{bench}_*.parquet')
)
if not per_model_files:
    raise FileNotFoundError(f'no result files in {RESULTS_DIR} — run the model sweep first.')

runs = [pd.read_parquet(p) for p in per_model_files]
runs = [run for run in runs if 'bench' in run.columns]
print('loaded', len(runs), 'runs')

summary_rows = []
for run in runs:
    bench = run['bench'].iloc[0]
    model = run['model'].iloc[0]
    row = summarize_run(
        f'A:{bench}:{model}',
        run['gold'].tolist(), run['pred'].tolist(), run['latency_ms'].tolist(),
    )
    row['bench'] = bench
    row['model'] = model
    summary_rows.append(row)

summary = (
    pd.DataFrame(summary_rows)
    .sort_values(['bench', 'macro_f1'], ascending=[True, False])
    .reset_index(drop=True)
)
summary

loaded 12 runs


,approach,macro_f1,n_scored,n_total,mean_ms,p50_ms,p95_ms,p99_ms,bench,model
0,A:hand_labelled:qwen2.5:3b,0.3374,197,197,665.5,643.0,850.6,944.1,hand_labelled,qwen2.5:3b
1,A:hand_labelled:gemma3:12b,0.2220,197,197,2262.9,2190.0,3020.4,3398.8,hand_labelled,gemma3:12b
2,A:hand_labelled:qwen2.5:14b-instruct,0.1734,197,197,1861.9,1737.0,2511.2,2767.5,hand_labelled,qwen2.5:14b-instruct
3,A:hand_labelled:qwen2.5:7b-instruct,0.1462,197,197,1045.0,932.0,1328.2,1436.1,hand_labelled,qwen2.5:7b-instruct
4,A:hand_labelled:phi4,0.1224,197,197,1901.4,1784.0,2526.8,2881.1,hand_labelled,phi4
5,A:hand_labelled:qwen3:8b,0.0000,197,197,3402.2,3288.0,3714.2,3868.6,hand_labelled,qwen3:8b
6,A:rule_based:qwen2.5:3b,0.7895,1023,1023,714.5,688.0,935.7,1125.3,rule_based,qwen2.5:3b
7,A:rule_based:gemma3:12b,0.7033,1023,1023,2175.4,2130.0,2796.9,3140.0,rule_based,gemma3:12b
8,A:rule_based:qwen2.5:7b-instruct,0.6750,1023,1023,1126.7,1083.0,1531.6,1750.5,rule_based,qwen2.5:7b-instruct
9,A:rule_based:qwen2.5:14b-instruct,0.6652,1023,1023,1935.3,1930.0,2419.8,2631.8,rule_based,qwen2.5:14b-instruct


In [7]:
# Per-class report for the best model within each benchmark.
if all_runs:
    for bench in sorted({r['bench'].iloc[0] for r in all_runs}):
        candidates = [r for r in all_runs if r['bench'].iloc[0] == bench]
        best = max(candidates, key=lambda r: summarize_run('x', r['gold'].tolist(), r['pred'].tolist(), [])['macro_f1'])
        best_model = best['model'].iloc[0]
        print(f'best for {bench}:', best_model)
        display(per_class_report(best['gold'].tolist(), best['pred'].tolist()))
        display(confusion_df(best['gold'].tolist(), best['pred'].tolist()))

best for hand_labelled: qwen2.5:3b


,category,precision,recall,f1,support
0,junk,0.1429,1.0000,0.2500,1
1,service_feedback,0.8727,0.3478,0.4974,138
2,config_feedback,0.2192,0.7273,0.3368,22
3,app_specific,0.2097,0.3611,0.2653,36
4,macro avg,0.3611,0.6091,0.3374,197
5,weighted avg,0.6749,0.3959,0.4358,197
6,accuracy,NaN,NaN,0.3959,197


,pred_junk,pred_service_feedback,pred_config_feedback,pred_app_specific,pred_others
true_junk,1,0,0,0,0
true_service_feedback,0,48,45,45,0
true_config_feedback,2,0,16,4,0
true_app_specific,4,7,12,13,0
true_others,0,0,0,0,0


best for rule_based: qwen2.5:3b


,category,precision,recall,f1,support
0,junk,0.9839,0.9919,0.9879,123
1,service_feedback,0.9512,0.5200,0.6724,300
2,config_feedback,0.5675,0.9667,0.7152,300
3,app_specific,0.9152,0.6833,0.7824,300
4,macro avg,0.8544,0.7905,0.7895,1023
5,weighted avg,0.8321,0.7556,0.7551,1023
6,accuracy,NaN,NaN,0.7556,1023


,pred_junk,pred_service_feedback,pred_config_feedback,pred_app_specific,pred_others
true_junk,122,0,0,1,0
true_service_feedback,1,156,131,12,0
true_config_feedback,0,4,290,6,0
true_app_specific,1,4,90,205,0
true_others,0,0,0,0,0


In [8]:
# Save combined for 06_evaluation
if all_runs:
    combined = pd.concat(all_runs, ignore_index=True)
    combined.to_parquet(RESULTS_DIR / 'approach_a_all.parquet', index=False)
    print('combined →', RESULTS_DIR / 'approach_a_all.parquet', len(combined))

combined → /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/results/approach_a_all.parquet 7320
